In [24]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from dotenv import load_dotenv
import os

load_dotenv()

True

In [25]:
llm1 = ChatOllama(model="nemotron-3-nano:30b-cloud")
llm2 = ChatGroq(model="llama-3.3-70b-versatile")
llm3 = ChatOllama(model="smollm2:360m")

In [26]:
final_model = llm1.with_fallbacks(fallbacks=[llm2,llm3])

In [27]:
MAX_TOKEN = 150

In [28]:
def call_model(state: MessagesState):
    #Trim convo history-> Last n message that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens= MAX_TOKEN
    )
    print("Current Token Count ->", count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)
    
    response = final_model.invoke(messages)
    return{
        "messages":[response]
    }

In [29]:
workflow = StateGraph(MessagesState)
workflow.add_node(call_model,"call_model")
workflow.add_edge(START,"call_model")

In [30]:
DB_URL= os.getenv("DB_URL")
if not DB_URL:
    raise ValueError("DATABASE_URL not found in environment or .env file!")

In [38]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # Run once (creates tables)
    checkpointer.setup()

    graph= workflow.compile(checkpointer=checkpointer)

    # Thread -1 
    t1 = {
        "configurable": {
        "thread_id":"chat-1"
        }
    }
    # result = graph.invoke({
    #     "messages":[
    #         {
    #             "role":"user",
    #             "content":"Hello my name is Suman"
    #         }
    #     ]
    # },t1)
    # print(result["messages"][-1].content)
    out1 = graph.invoke({
        "messages":[
            {
                "role":"user",
                "content":"Can you explain Long term memory"
            }
        ]
    },t1)

    print("Thread-1",out1["messages"][-1].content)

    # result2 = graph.invoke({
    #     "messages":[
    #         {
    #             "role":"user",
    #             "content":"Can you explain Short term memory"
    #         }
    #     ]
    # },t1)
    # print(result2["messages"][-1].content)

Current Token Count -> 12
Can you explain Long term memory


KeyboardInterrupt: 